# Indian Wildfire Risk Model Training (XGBoost)
**Dataset**: NASA FIRMS India Satellite Fire Archive (8 Years)
**Goal**: Predict wildfire occurrence and intensity with high accuracy

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
print("Libraries loaded successfully!")


In [ ]:
import glob
import os
print("Available input directories:", os.listdir("/kaggle/input"))
matching = glob.glob("/kaggle/input/**/fire_archive_M6_*.csv", recursive=True) or glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("Found files:", matching)
if not matching:
    raise FileNotFoundError("No FIRMS CSV found in /kaggle/input")
data_path = matching[0]
print(f"Reading NASA FIRMS dataset from: {data_path}")
df = pd.read_csv(data_path, low_memory=False)
print(f"Raw dataset shape: {df.shape}")
print(df.head())


In [ ]:
# Filter for presumed vegetation wildfires (type == 0)
if "type" in df.columns:
    df = df[df["type"] == 0].copy()

# Parse date features
df["acq_date"] = pd.to_datetime(df["acq_date"])
df["year"] = df["acq_date"].dt.year
df["month"] = df["acq_date"].dt.month
df["day_of_year"] = df["acq_date"].dt.dayofyear
df["day_of_week"] = df["acq_date"].dt.dayofweek
df["is_day"] = (df["daynight"] == "D").astype(int)

# Target: High-confidence significant wildfire detection
df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce").fillna(0)
df["frp"] = pd.to_numeric(df["frp"], errors="coerce").fillna(0)
df["target"] = ((df["confidence"] >= 50) & (df["frp"] >= 10.0)).astype(int)

print(f"Processed dataset shape: {df.shape}")
print(f"Target distribution: {df['target'].value_counts(normalize=True)}")


In [ ]:
features = [
    "latitude", "longitude", "brightness", "scan", "track",
    "bright_t31", "month", "day_of_year", "day_of_week", "is_day"
]

# Time-based train / validation / test split
train_mask = df["year"] <= 2017
val_mask = df["year"] == 2018
test_mask = df["year"] >= 2019

X_train, y_train = df.loc[train_mask, features], df.loc[train_mask, "target"]
X_val, y_val = df.loc[val_mask, features], df.loc[val_mask, "target"]
X_test, y_test = df.loc[test_mask, features], df.loc[test_mask, "target"]

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")


In [ ]:
print("Training XGBoost Classifier on NASA FIRMS historical data...")
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

print("=" * 50)
print(f"TEST ACCURACY: {acc * 100:.2f}%")
print(f"TEST ROC-AUC:  {roc:.4f}")
print(f"TEST F1-SCORE: {f1:.4f}")
print(f"TEST PRECISION:{prec:.4f}")
print(f"TEST RECALL:   {rec:.4f}")
print("=" * 50)
print(classification_report(y_test, y_pred))


In [ ]:
# Export trained model and metrics to working directory
model.save_model("/kaggle/working/wildfire_xgb_v1.json")
metrics = {
    "model_name": "wildfire_xgb_v1",
    "accuracy": round(acc, 4),
    "roc_auc": round(roc, 4),
    "f1_score": round(f1, 4),
    "precision": round(prec, 4),
    "recall": round(rec, 4),
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "features": features
}
with open("/kaggle/working/wildfire_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved wildfire_xgb_v1.json and wildfire_metrics.json to /kaggle/working/")
